In [0]:
# DataSentry v2 reset utility
# Drops TABLES only (never Volumes, never v1). Choose layers and dataset below.
# v1 (banking_datasentry) is deliberately not in the allowed list.

CATALOG = "workspace"
ALLOWED_SCHEMAS = ["dq_bronze", "dq_silver", "dq_gold", "dq_config"]

# What to reset: any of "dq_bronze", "dq_silver", "dq_gold", "dq_config"
LAYERS_TO_RESET = ["dq_silver", "dq_gold"]

# Limit to one dataset's tables (e.g. "banking"), or None for every table in those layers
DATASET = None

In [0]:
# Preview: list what would be dropped. Nothing is deleted here.
spark.sql(f"USE CATALOG {CATALOG}")

existing = {r[0] for r in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()}

to_drop = []
for schema in LAYERS_TO_RESET:
    assert schema in ALLOWED_SCHEMAS, f"Refusing to touch schema '{schema}'"
    if schema not in existing:
        print(f"Skipping {schema}: schema does not exist yet")
        continue
    for t in spark.sql(f"SHOW TABLES IN {schema}").collect():
        if t.isTemporary:
            continue
        if DATASET is None or t.tableName.startswith(f"{DATASET}_"):
            to_drop.append(f"{CATALOG}.{schema}.{t.tableName}")

print(f"\n{len(to_drop)} tables would be dropped:")
for name in to_drop:
    print("  ", name)

In [0]:
# Drop: run only after checking the preview above
for name in to_drop:
    spark.sql(f"DROP TABLE IF EXISTS {name}")
    print("Dropped", name)

print(f"\nDone. {len(to_drop)} tables dropped. Volumes and files untouched.")